# Spotify Genre Classification with Deep Learning
**AI 100 Midterm Project — Marco King**

Classifying Spotify tracks into 10 super-genres using a PyTorch MLP trained on audio features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Load Dataset

In [ ]:
df = pd.read_csv('dataset.csv', encoding='utf-8')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Unique genres: {df['track_genre'].nunique()}")
df.head()

## 2. Map 114 Genres to 10 Super-Genres

In [ ]:
genre_map = {
    'Rock': ['rock', 'alt-rock', 'alternative', 'hard-rock', 'punk', 'punk-rock',
             'grunge', 'psych-rock', 'rock-n-roll', 'rockabilly', 'emo', 'goth', 'indie', 'ska'],
    'Metal': ['metal', 'heavy-metal', 'death-metal', 'black-metal', 'metalcore',
              'grindcore', 'hardcore', 'industrial'],
    'Pop': ['pop', 'indie-pop', 'synth-pop', 'k-pop', 'j-pop', 'cantopop',
            'mandopop', 'j-idol', 'pop-film', 'power-pop', 'british', 'swedish'],
    'Electronic': ['edm', 'electro', 'electronic', 'house', 'chicago-house',
                   'deep-house', 'detroit-techno', 'techno', 'minimal-techno',
                   'progressive-house', 'trance', 'hardstyle', 'drum-and-bass',
                   'dubstep', 'breakbeat', 'garage', 'idm', 'trip-hop', 'dub'],
    'Hip-Hop/R&B': ['hip-hop', 'r-n-b', 'reggaeton', 'dancehall'],
    'Jazz/Blues': ['jazz', 'blues', 'soul', 'funk', 'groove'],
    'Classical': ['classical', 'piano', 'opera', 'guitar', 'new-age', 'ambient',
                  'sleep', 'study'],
    'Folk/Country': ['folk', 'country', 'acoustic', 'bluegrass', 'honky-tonk',
                     'singer-songwriter', 'songwriter'],
    'Latin/World': ['latin', 'latino', 'salsa', 'samba', 'forro', 'sertanejo',
                    'pagode', 'mpb', 'brazil', 'afrobeat', 'indian', 'iranian',
                    'turkish', 'malay', 'tango', 'reggae', 'french', 'german',
                    'spanish', 'world-music'],
    'Dance/Other': ['dance', 'club', 'disco', 'party', 'happy', 'chill', 'romance',
                    'sad', 'anime', 'disney', 'children', 'kids', 'comedy',
                    'show-tunes', 'j-dance', 'j-rock', 'gospel']
}

# Invert: original_genre -> super_genre
reverse_map = {}
for super_genre, originals in genre_map.items():
    for g in originals:
        reverse_map[g] = super_genre

df['super_genre'] = df['track_genre'].map(reverse_map)

# Check for unmapped genres
unmapped = df[df['super_genre'].isna()]['track_genre'].unique()
print(f"Unmapped genres: {unmapped}")
print(f"\nSuper-genre distribution:")
print(df['super_genre'].value_counts())

## 3. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
df['super_genre'].value_counts().plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Distribution of Super-Genres', fontsize=14)
ax.set_xlabel('Super-Genre')
ax.set_ylabel('Number of Tracks')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()

In [ ]:
audio_features = ['danceability', 'energy', 'key', 'loudness', 'mode',
                  'speechiness', 'acousticness', 'instrumentalness',
                  'liveness', 'valence', 'tempo', 'time_signature', 'duration_ms']

fig, ax = plt.subplots(figsize=(12, 10))
corr = df[audio_features].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Audio Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150)
plt.show()

In [ ]:
key_features = ['danceability', 'energy', 'acousticness', 'instrumentalness', 'valence', 'speechiness']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, feat in enumerate(key_features):
    ax = axes[i // 3][i % 3]
    df.boxplot(column=feat, by='super_genre', ax=ax, rot=90)
    ax.set_title(feat.capitalize())
    ax.set_xlabel('')
plt.suptitle('Audio Feature Distributions by Super-Genre', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150)
plt.show()

## 4. Data Preprocessing

In [ ]:
X = df[audio_features].values
le = LabelEncoder()
y = le.fit_transform(df['super_genre'].values)

print(f"Features shape: {X.shape}")
print(f"Classes: {le.classes_}")
print(f"Number of classes: {len(le.classes_)}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain set: {X_train.shape[0]} samples")
print(f"Test set:  {X_test.shape[0]} samples")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_scaled).to(device)
y_train_tensor = torch.LongTensor(y_train).to(device)
X_test_tensor = torch.FloatTensor(X_test_scaled).to(device)
y_test_tensor = torch.LongTensor(y_test).to(device)

# Create DataLoaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print("Data ready for training.")

## 5. Baseline Model (Logistic Regression)

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
lr_acc = accuracy_score(y_test, lr_pred)

print(f"Logistic Regression Accuracy: {lr_acc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, lr_pred, target_names=le.classes_))

## 6. Deep Learning Model — MLP (PyTorch)

In [ ]:
class GenreClassifier(nn.Module):
    def __init__(self, input_size, num_classes):
        super(GenreClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.network(x)

input_size = X_train_scaled.shape[1]  # 13 features
num_classes = len(le.classes_)         # 10 super-genres

model = GenreClassifier(input_size, num_classes).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## 7. Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

num_epochs = 50
train_losses = []
train_accs = []
test_accs = []

for epoch in range(num_epochs):
    # Training
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total
    train_losses.append(epoch_loss)
    train_accs.append(epoch_acc)

    # Evaluation
    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test_tensor)
        _, test_pred = torch.max(test_outputs, 1)
        test_acc = (test_pred == y_test_tensor).sum().item() / y_test_tensor.size(0)
        test_accs.append(test_acc)

    scheduler.step(epoch_loss)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {epoch_loss:.4f} "
              f"Train Acc: {epoch_acc:.4f} Test Acc: {test_acc:.4f}")

print(f"\nFinal Test Accuracy: {test_accs[-1]:.4f}")

## 8. Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, label='Training Loss', color='steelblue')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss Over Epochs')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(train_accs, label='Train Accuracy', color='steelblue')
ax2.plot(test_accs, label='Test Accuracy', color='coral')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy Over Epochs')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

In [ ]:
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, y_pred = torch.max(test_outputs, 1)
    y_pred_np = y_pred.cpu().numpy()

cm = confusion_matrix(y_test, y_pred_np)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix — MLP Genre Classifier', fontsize=14)
plt.tight_layout()
plt.savefig('confusion_matrix_mlp.png', dpi=150)
plt.show()

In [ ]:
print("=" * 60)
print("MLP Classification Report")
print("=" * 60)
print(classification_report(y_test, y_pred_np, target_names=le.classes_))

print("\n" + "=" * 60)
print("Model Comparison")
print("=" * 60)
mlp_acc = accuracy_score(y_test, y_pred_np)
print(f"Logistic Regression: {lr_acc:.4f}")
print(f"MLP Neural Network:  {mlp_acc:.4f}")
print(f"Improvement:         {mlp_acc - lr_acc:+.4f} ({(mlp_acc - lr_acc) / lr_acc * 100:+.1f}%)")

## Summary

- **Problem:** Classify Spotify tracks into 10 super-genres using audio features
- **Dataset:** ~114,000 tracks with 13 audio features from Spotify
- **Baseline:** Logistic Regression
- **Model:** 3-layer MLP with BatchNorm and Dropout (PyTorch)
- **Key Finding:** The MLP outperformed logistic regression, demonstrating the value of deep learning for capturing non-linear patterns in audio feature data.
- **Challenge:** Some super-genres (e.g., Dance/Other, Pop) overlap significantly in audio feature space, making perfect classification difficult.